## Tugas 7

In [ ]:
import openeo
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
processes = connection.list_processes()
print([p["id"] for p in processes[:10]])

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [ 112.7081925258214, -7.114976340860011 ],
            [ 112.7081925258214, -7.139727758223174 ],
            [ 112.74157115746493, -7.139727758223174 ],
            [ 112.74157115746493, -7.114976340860011 ],
            [ 112.7081925258214, -7.114976340860011 ]
        ]
    ],
}

In [ ]:
s2_during = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=["2020-06-01", "2021-06-30"],
    spatial_extent={"west": 112.7081925258214, "south": -7.139727758223174, 
                    "east": 112.74157115746493, "north": -7.114976340860011},
    bands=["B04", "B08"]
).ndvi("B08", "B04")

# Agregasi temporal ke daily mean (hindari multiple data per hari)
s2_during = s2_during.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial: mean over exact polygon AOI
s2_during = s2_during.aggregate_spatial(reducer="mean", geometries=aoi)

In [ ]:
s2_post = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=["2022-06-01", "2023-06-30"],
    spatial_extent={"west": 112.7081925258214, "south": -7.139727758223174, 
                    "east": 112.74157115746493, "north": -7.114976340860011},
    bands=["B04", "B08"]
).ndvi("B08", "B04")

# Agregasi temporal ke daily mean
s2_post = s2_post.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial: mean over exact polygon AOI
s2_post = s2_post.aggregate_spatial(reducer="mean", geometries=aoi)

In [ ]:
# Eksekusi batch jobs (jalan di backend Copernicus)
job_during = s2_during.execute_batch(title="NDVI During COVID Malang", outputfile="ndvi_during_covid.nc")
job_post = s2_post.execute_batch(title="NDVI Post-COVID Malang", outputfile="ndvi_post_covid.nc")

In [ ]:
# Load hasil ke xarray
during_data = xr.load_dataset("ndvi_during_covid.nc")
post_data = xr.load_dataset("ndvi_post_covid.nc")

In [ ]:
# Rolling mean 30 hari untuk smooth timeseries
during_data = during_data.rolling(t=30).mean()
post_data = post_data.rolling(t=30).mean()

In [ ]:
fig, ax1 = plt.subplots(dpi=100, figsize=(12, 6))
(line1,) = ax1.plot(
    during_data.t, during_data.NDVI.to_numpy().flatten(), color="r", label="During COVID", linewidth=2
)
ax1.set_xlabel("During COVID (2020-2021)")
ax1.set_ylabel("NDVI Level (Vegetasi)")
ax1.set_title("Perbandingan NDVI Malang: Dampak COVID Lockdown")
ax1.xaxis.label.set_color("r")
ax1.tick_params(axis="x", colors="r")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twiny()
(line2,) = ax2.plot(
    post_data.t, post_data.NDVI.to_numpy().flatten(), color="g", label="Post COVID", linewidth=2
)
ax2.set_xlabel("Post COVID (2022-2023)")
# Gabung legend
lines = [line1, line2]
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="upper left")
ax2.xaxis.label.set_color("g")
ax2.tick_params(axis="x", colors="g")

plt.tight_layout()
plt.show()

In [ ]:
# pip install openeo xarray netcdf4 matplotlib

In [ ]:
# import openeo
# import xarray as xr
# import matplotlib.pyplot as plt
# import logging  # Untuk debug

# # Enable logging untuk debug koneksi (opsional, hapus kalau nggak perlu)
# logging.basicConfig(level=logging.DEBUG)

# # Opsi: True untuk pakai VITO backend (workaround kalau Copernicus error)
# use_vito = False  # Set True kalau RemoteDisconnected persist

# if use_vito:
#     # Workaround: Backend VITO (stabil, data global termasuk Indonesia)
#     connection = openeo.connect("https://openeo.vito.be/1.0").authenticate_oidc()
#     print("Menggunakan backend VITO (workaround).")
# else:
#     # Default: Copernicus
#     connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
#     print("Menggunakan backend Copernicus.")

# # Test koneksi
# processes = connection.list_processes()
# print("Koneksi sukses! Sample processes:", [p["id"] for p in processes[:5]])

# # AOI polygon Malang (koordinat fixed: semua lat negatif)
# aoi = {
#     "type": "Polygon",
#     "coordinates": [
#         [
#             [112.7081925258214, -7.114976340860011],
#             [112.7081925258214, -7.139727758223174],
#             [112.74157115746493, -7.139727758223174],
#             [112.74157115746493, -7.114976340860011],
#             [112.7081925258214, -7.114976340860011]
#         ]
#     ],
# }

# # Bbox untuk load collection (efisien)
# bbox = {
#     "west": 112.7081925258214,
#     "south": -7.139727758223174,
#     "east": 112.74157115746493,
#     "north": -7.114976340860011
# }

# # Fungsi helper untuk cloud masking (SCL < 8: non-awan)
# def cloud_mask_scl(cube):
#     return cube.filter_bands("SCL").apply(lambda x: x.array_mask(x.SCL < 8)).drop_bands("SCL")

# # Load Sentinel-2 L2A during COVID + mask + NDVI
# print("Memuat data during COVID...")
# s2_during = connection.load_collection(
#     "SENTINEL2_L2A",
#     temporal_extent=["2020-06-01", "2021-06-30"],
#     spatial_extent=bbox,
#     bands=["B04", "B08", "SCL"]
# )
# s2_during = cloud_mask_scl(s2_during).filter_bands(["B04", "B08"]).ndvi("B08", "B04")

# # Agregasi temporal: daily mean
# s2_during = s2_during.aggregate_temporal_period(reducer="mean", period="day")

# # Agregasi spasial: mean over exact polygon
# s2_during = s2_during.aggregate_spatial(reducer="mean", geometries=aoi)

# # Load post-COVID (sama)
# print("Memuat data post-COVID...")
# s2_post = connection.load_collection(
#     "SENTINEL2_L2A",
#     temporal_extent=["2022-06-01", "2023-06-30"],
#     spatial_extent=bbox,
#     bands=["B04", "B08", "SCL"]
# )
# s2_post = cloud_mask_scl(s2_post).filter_bands(["B04", "B08"]).ndvi("B08", "B04")

# # Agregasi temporal
# s2_post = s2_post.aggregate_temporal_period(reducer="mean", period="day")

# # Agregasi spasial
# s2_post = s2_post.aggregate_spatial(reducer="mean", geometries=aoi)

# # Eksekusi batch jobs (wait sampai selesai)
# print("Memulai job during COVID...")
# job_during = s2_during.execute_batch(title="NDVI During COVID Malang", outputfile="ndvi_during_covid.nc")
# job_during.start_and_wait()  # Wait ~10-30 menit, monitor progress

# print("Memulai job post-COVID...")
# job_post = s2_post.execute_batch(title="NDVI Post-COVID Malang", outputfile="ndvi_post_covid.nc")
# job_post.start_and_wait()

# # Load hasil ke xarray
# print("Memuat hasil untuk plotting...")
# during_data = xr.load_dataset("ndvi_during_covid.nc")
# post_data = xr.load_dataset("ndvi_post_covid.nc")

# # Rolling mean 30 hari untuk smooth
# during_data = during_data.rolling(t=30).mean()
# post_data = post_data.rolling(t=30).mean()

# # Plot timeseries
# fig, ax1 = plt.subplots(dpi=100, figsize=(12, 6))
# (line1,) = ax1.plot(
#     during_data.t, during_data.NDVI.to_numpy().flatten(), color="r", label="During COVID", linewidth=2
# )
# ax1.set_xlabel("During COVID (2020-2021)")
# ax1.set_ylabel("NDVI Level (Vegetasi)")
# ax1.set_title("Perbandingan NDVI Malang: Dampak COVID Lockdown")
# ax1.xaxis.label.set_color("r")
# ax1.tick_params(axis="x", colors="r")
# ax1.grid(True, alpha=0.3)

# ax2 = ax1.twiny()
# (line2,) = ax2.plot(
#     post_data.t, post_data.NDVI.to_numpy().flatten(), color="g", label="Post COVID", linewidth=2
# )
# ax2.set_xlabel("Post COVID (2022-2023)")
# lines = [line1, line2]
# labels = [line.get_label() for line in lines]
# ax1.legend(lines, labels, loc="upper left")
# ax2.xaxis.label.set_color("g")
# ax2.tick_params(axis="x", colors="g")

# plt.tight_layout()
# plt.savefig("ndvi_plot_malang.png")  # Simpan plot sebagai PNG
# plt.show()

# print("Selesai! Cek file: ndvi_during_covid.nc, ndvi_post_covid.nc, dan ndvi_plot_malang.png")